In [ ]:
import os
import pymc

In [ ]:
import random
from collections import Counter

import pandas as pd
from datasets import load_dataset


# -----------------------------
# Reproducibility for random tie-breaks
# -----------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


# -----------------------------
# Load datasets
# -----------------------------
print("Loading datasets...")

personas = load_dataset(
    "thoughtworks/psychometric_personas",
    "analysis",
    split="train"
).to_pandas()

sjts = load_dataset(
    "thoughtworks/psychometric_sjts_analysis",
    "analysis",
    split="train"
).to_pandas()

responses = load_dataset(
    "thoughtworks/gemma_psychometrics_personas_responses",
    "analysis_sjt",
    split="train"
).to_pandas()

print("Datasets loaded:")
print("  Personas :", len(personas))
print("  SJTs     :", len(sjts))
print("  Responses:", len(responses))


# -----------------------------
# Normalize key names
# -----------------------------
# In the SJT dataset, the question hash column is called `hash_id`.
# Rename it so all three datasets join consistently on `question_hash`.
if "hash_id" in sjts.columns:
    sjts = sjts.rename(columns={"hash_id": "question_hash"})

required_persona_cols = {"persona_hash"}
required_sjt_cols = {"question_hash"}
required_response_cols = {"persona_hash", "question_hash", "normalized_answer"}

missing_persona = required_persona_cols - set(personas.columns)
missing_sjt = required_sjt_cols - set(sjts.columns)
missing_response = required_response_cols - set(responses.columns)

if missing_persona:
    raise ValueError(f"Personas dataset missing required columns: {missing_persona}")
if missing_sjt:
    raise ValueError(f"SJTs dataset missing required columns: {missing_sjt}")
if missing_response:
    raise ValueError(f"Responses dataset missing required columns: {missing_response}")


# -----------------------------
# Verify expected counts
# -----------------------------
expected_responses = 500 * 300 * 5
print("\nExpected responses:", expected_responses)
if len(responses) != expected_responses:
    print(f"WARNING: expected {expected_responses}, got {len(responses)}")

expected_pairs = 500 * 300


# -----------------------------
# Mode calculation with tie detection
# -----------------------------
print("\nComputing per-(persona, question) modal response...")

tie_count = 0
rows = []

grouped = responses.groupby(["persona_hash", "question_hash"], sort=False)

for (persona_hash, question_hash), group in grouped:
    answers = group["normalized_answer"].tolist()
    counts = Counter(answers)
    max_count = max(counts.values())
    modes = [ans for ans, cnt in counts.items() if cnt == max_count]

    is_tie = len(modes) > 1
    if is_tie:
        tie_count += 1

    final_answer = random.choice(modes)

    rows.append({
        "persona_hash": persona_hash,
        "question_hash": question_hash,
        "answer": final_answer,
        "is_tie": is_tie,
        "n_unique_answers_among_5": len(counts),
        "mode_count": max_count,
        "all_answers_5": answers,
    })

final_df = pd.DataFrame(rows)

print("Final response rows:", len(final_df))
print("Expected persona x question pairs:", expected_pairs)
if len(final_df) != expected_pairs:
    print(f"WARNING: expected {expected_pairs}, got {len(final_df)}")


# -----------------------------
# Tie statistics
# -----------------------------
total_pairs = len(final_df)
tie_rate = tie_count / total_pairs if total_pairs > 0 else 0.0

print("\nTie statistics")
print("---------------------------")
print("Total persona-question pairs:", total_pairs)
print("Pairs with ties            :", tie_count)
print("Tie rate                   :", f"{tie_rate:.6f}")


# Optional: breakdown of tie types
if tie_count > 0:
    print("\nTie breakdown by mode_count:")
    print(final_df.loc[final_df["is_tie"], "mode_count"].value_counts().sort_index())


# -----------------------------
# Join metadata
# -----------------------------
print("\nJoining persona + question metadata...")

final_df = final_df.merge(
    personas,
    on="persona_hash",
    how="left",
    validate="many_to_one"
)

final_df = final_df.merge(
    sjts,
    on="question_hash",
    how="left",
    validate="many_to_one"
)

print("Final merged dataset size:", len(final_df))


# -----------------------------
# Check join quality
# -----------------------------
missing_persona_meta = final_df["persona_hash"].isna().sum()
missing_question_meta = final_df["question_hash"].isna().sum()

print("\nJoin checks")
print("---------------------------")
print("Missing persona_hash after join :", missing_persona_meta)
print("Missing question_hash after join:", missing_question_meta)


# -----------------------------
# Save clean long-form dataset
# -----------------------------
long_outfile = "persona_sjt_responses_clean.parquet"
final_df.to_parquet(long_outfile, index=False)
print(f"\nSaved: {long_outfile}")


# -----------------------------
# Create MIRT matrix
# -----------------------------
print("\nConstructing response matrix for MIRT...")

matrix = final_df.pivot(
    index="persona_hash",
    columns="question_hash",
    values="answer"
)

print("Matrix shape:", matrix.shape)

matrix_outfile = "mirt_response_matrix.parquet"
matrix.to_parquet(matrix_outfile)
print(f"Saved: {matrix_outfile}")


# -----------------------------
# Optional CSV exports
# -----------------------------
final_df[[
    "persona_hash",
    "question_hash",
    "answer",
    "is_tie",
    "mode_count",
    "n_unique_answers_among_5"
]].to_csv("persona_sjt_responses_for_mirt.csv", index=False)

print("Saved: persona_sjt_responses_for_mirt.csv")

In [ ]:
import pandas as pd
import numpy as np

# Load collapsed data
df = pd.read_parquet("persona_sjt_responses_clean.parquet")

print("Rows:", len(df))
print("Unique personas:", df["persona_hash"].nunique())
print("Unique questions:", df["question_hash"].nunique())

# -----------------------------
# 1. Check exactly one row per persona-question
# -----------------------------
pair_counts = (
    df.groupby(["persona_hash", "question_hash"])
      .size()
      .value_counts()
      .sort_index()
)

print("\nRows per (persona, question) pair:")
print(pair_counts)

# -----------------------------
# 2. Inspect answer space
# -----------------------------
answers = sorted(df["answer"].dropna().unique().tolist())
print("\nUnique final answers:")
print(answers)
print("Num unique answers:", len(answers))

# -----------------------------
# 3. Tie summaries
# -----------------------------
print("\nTie summary:")
print(df["is_tie"].value_counts(dropna=False))
print("\nMode count summary:")
print(df["mode_count"].value_counts(dropna=False).sort_index())

# -----------------------------
# 4. Create deterministic answer coding
# -----------------------------
# IMPORTANT:
# This uses the observed lexical order of answer labels.
# If your 6 normalized answers correspond to specific HEXACO dimensions,
# you may want to replace this with a semantically meaningful order instead.
answer_to_int = {ans: i + 1 for i, ans in enumerate(answers)}
int_to_answer = {v: k for k, v in answer_to_int.items()}

print("\nAnswer encoding:")
print(answer_to_int)

df["response_int"] = df["answer"].map(answer_to_int)

if df["response_int"].isna().any():
    raise ValueError("Some answers were not encoded.")

# -----------------------------
# 5. Build integer response matrix
# -----------------------------
response_matrix = df.pivot(
    index="persona_hash",
    columns="question_hash",
    values="response_int"
).sort_index().sort_index(axis=1)

print("\nInteger response matrix shape:", response_matrix.shape)
print("Missing values in matrix:", int(response_matrix.isna().sum().sum()))

# -----------------------------
# 6. Save MIRT-ready artifacts
# -----------------------------
response_matrix.to_csv("mirt_response_matrix_int.csv")
df.to_parquet("persona_sjt_responses_clean_with_int.parquet", index=False)

encoding_df = pd.DataFrame({
    "answer": list(answer_to_int.keys()),
    "response_int": list(answer_to_int.values())
})
encoding_df.to_csv("answer_encoding.csv", index=False)

print("\nSaved:")
print("- mirt_response_matrix_int.csv")
print("- persona_sjt_responses_clean_with_int.parquet")
print("- answer_encoding.csv")

In [ ]:
persona_ties = (
    df.groupby("persona_hash")["is_tie"]
      .sum()
      .sort_values(ascending=False)
)

question_ties = (
    df.groupby("question_hash")["is_tie"]
      .sum()
      .sort_values(ascending=False)
)

print("Top 10 personas by tie count:")
print(persona_ties.head(10))

print("\nTop 10 questions by tie count:")
print(question_ties.head(10))

# Extreme instability: mode_count == 1
full_instability = df[df["mode_count"] == 1]

print("\nFull-instability rows:", len(full_instability))

print("\nTop 10 personas with mode_count == 1:")
print(full_instability["persona_hash"].value_counts().head(10))

print("\nTop 10 questions with mode_count == 1:")
print(full_instability["question_hash"].value_counts().head(10))

In [ ]:
import pandas as pd

encoding = pd.read_csv("answer_encoding.csv")
print(encoding)

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az

# ============================================================
# Cell 2: Confirmatory HEXACO MIRT (fast pilot)
# ============================================================

# -----------------------------
# 1. Load data
# -----------------------------
df = pd.read_parquet("persona_sjt_responses_clean_with_int.parquet").copy()

if df["response_int"].min() == 1:
    df["y"] = df["response_int"] - 1
else:
    df["y"] = df["response_int"]

assert df["y"].min() == 0
assert df["y"].max() == 5

# -----------------------------
# 2. Aggressive subsampling
# -----------------------------
N_PERSONS_PILOT = 250
N_ITEMS_PILOT = 150
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

keep_personas = rng.choice(
    df["persona_hash"].drop_duplicates().to_numpy(),
    size=N_PERSONS_PILOT,
    replace=False,
)

df = df[df["persona_hash"].isin(keep_personas)].copy()

keep_items = rng.choice(
    df["question_hash"].drop_duplicates().to_numpy(),
    size=N_ITEMS_PILOT,
    replace=False,
)

df = df[df["question_hash"].isin(keep_items)].copy()

# -----------------------------
# 3. Encode indices
# -----------------------------
df["person_idx"], person_index = pd.factorize(df["persona_hash"], sort=True)
df["item_idx"], item_index = pd.factorize(df["question_hash"], sort=True)

N = len(df)
P = df["person_idx"].nunique()
J = df["item_idx"].nunique()
K = 6
D = 6

print(f"N={N}, P={P}, J={J}, K={K}, D={D}")

person_idx = df["person_idx"].to_numpy(dtype="int32")
item_idx = df["item_idx"].to_numpy(dtype="int32")
y_obs = df["y"].to_numpy(dtype="int32")

# -----------------------------
# 4. Factor structure
# -----------------------------
# category order:
# 0 -> A
# 1 -> C
# 2 -> E
# 3 -> X
# 4 -> H
# 5 -> O

factor_names = ["A", "C", "E", "X", "H", "O"]

M = np.eye(D, dtype=np.float32)

coords = {
    "person": np.arange(P),
    "item": np.arange(J),
    "category": np.arange(K),
    "factor": factor_names,
    "obs": np.arange(N),
}

# -----------------------------
# 5. Model
# -----------------------------
with pm.Model(coords=coords) as hexaco_model:

    # persona latent traits
    theta = pm.Normal(
        "theta",
        mu=0,
        sigma=1,
        dims=("person", "factor"),
    )

    # item discrimination
    a = pm.HalfNormal(
        "a",
        sigma=1.0,
        dims="item",
    )

    # item-category intercepts
    alpha_raw = pm.Normal(
        "alpha_raw",
        mu=0,
        sigma=1,
        dims=("item", "category"),
    )

    alpha = pm.Deterministic(
        "alpha",
        alpha_raw - alpha_raw.mean(axis=1, keepdims=True),
        dims=("item", "category"),
    )

    theta_obs = theta[person_idx]
    alpha_obs = alpha[item_idx]
    a_obs = a[item_idx][:, None]

    category_trait_score = theta_obs @ M.T

    eta = alpha_obs + a_obs * category_trait_score

    y = pm.Categorical(
        "y",
        logit_p=eta,
        observed=y_obs,
        dims="obs",
    )

    idata_hexaco = pm.sample(
        draws=300,
        tune=300,
        chains=8,
        cores=8,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
    )

# -----------------------------
# 6. Save results
# -----------------------------
az.to_netcdf(idata_hexaco, "hexaco_confirmatory_mirt_pilot.nc")

print("Saved: hexaco_confirmatory_mirt_pilot.nc")

# -----------------------------
# 7. Extract item discrimination
# -----------------------------
a_mean = (
    idata_hexaco.posterior["a"]
    .mean(dim=("chain", "draw"))
    .to_pandas()
)

a_df = pd.DataFrame({
    "question_hash": item_index,
    "a_mean": a_mean.values,
})

a_df.to_csv("hexaco_item_discrimination_pilot.csv", index=False)

print("Saved: hexaco_item_discrimination_pilot.csv")

# -----------------------------
# 8. Posterior summary
# -----------------------------
print("\nPosterior summary:")
print(
    az.summary(
        idata_hexaco,
        var_names=["a"],
        round_to=3,
    )
)

In [ ]:
summary = az.summary(idata_hexaco, var_names=None)
print(summary[["r_hat", "ess_bulk", "ess_tail"]])

In [ ]:
import pandas as pd
import arviz as az

# get summary
summary = az.summary(idata_hexaco)

rhat = summary["r_hat"]

# fractions
total = len(rhat)
in_range = ((rhat >= 1.00) & (rhat <= 1.01)).sum()
below_101 = (rhat <= 1.01).sum()
above_101 = (rhat > 1.01).sum()

table = pd.DataFrame({
    "metric": [
        "Total parameters",
        "1.00 ≤ R̂ ≤ 1.01",
        "R̂ ≤ 1.01",
        "R̂ > 1.01"
    ],
    "count": [
        total,
        in_range,
        below_101,
        above_101
    ],
    "fraction": [
        1.0,
        in_range / total,
        below_101 / total,
        above_101 / total
    ]
})

print(table)

In [ ]:
summary_df = az.summary(
    idata_hexaco,
    var_names=["a"],
    round_to=3,
)

summary_df.to_json(
    "hexaco_item_discrimination_summary.json",
    orient="index",
    indent=2,
)

print("Saved: hexaco_item_discrimination_summary.json")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# Load ArviZ summary JSON
# ============================================================

with open("hexaco_item_discrimination_summary.json", "r") as f:
    summary = json.load(f)

df = (
    pd.DataFrame.from_dict(summary, orient="index")
    .reset_index()
    .rename(columns={"index": "item"})
)

df = df[df["item"].str.startswith("a[")].copy()

a = df["mean"].to_numpy()

# ============================================================
# Core descriptive statistics
# ============================================================

stats = {
    "n_items": int(len(a)),
    "mean": float(np.mean(a)),
    "median": float(np.median(a)),
    "std": float(np.std(a, ddof=1)),
    "min": float(np.min(a)),
    "max": float(np.max(a)),
    "q10": float(np.quantile(a, 0.10)),
    "q25": float(np.quantile(a, 0.25)),
    "q75": float(np.quantile(a, 0.75)),
    "q90": float(np.quantile(a, 0.90)),
}

print("\nDiscrimination summary statistics")
for k,v in stats.items():
    print(f"{k}: {v:.4f}" if isinstance(v,float) else f"{k}: {v}")

# ============================================================
# Formal bucketed distribution
# ============================================================

bins = [-np.inf, 0.5, 1.0, 2.0, 3.0, np.inf]
labels = [
    "very_weak(<0.5)",
    "weak_to_moderate(0.5-1.0)",
    "good(1.0-2.0)",
    "strong(2.0-3.0)",
    "very_strong(>=3.0)"
]

df["bucket"] = pd.cut(
    df["mean"],
    bins=bins,
    labels=labels,
    right=False
)

distribution_table = (
    df["bucket"]
    .value_counts()
    .reindex(labels)
)

distribution_percent = 100 * distribution_table / len(df)

dist_df = pd.DataFrame({
    "count": distribution_table,
    "percent": distribution_percent.round(2)
})

print("\nBucketed distribution")
print(dist_df)

# Save tables
dist_df.to_csv("hexaco_discrimination_distribution.csv")
df.to_csv("hexaco_discrimination_full_table.csv", index=False)

# ============================================================
# Histogram visualization
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    a,
    bins=20,
    edgecolor="black",
    alpha=0.8
)

plt.axvline(np.mean(a), linestyle="--", label="mean")
plt.axvline(np.median(a), linestyle=":", label="median")

plt.xlabel("Item discrimination (a)")
plt.ylabel("Number of SJT items")
plt.title("Distribution of SJT Item Discrimination Parameters")
plt.legend()

plt.tight_layout()
plt.savefig("hexaco_discrimination_histogram.png", dpi=300)

plt.show()

print("\nSaved:")
print("- hexaco_discrimination_distribution.csv")
print("- hexaco_discrimination_full_table.csv")
print("- hexaco_discrimination_histogram.png")

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# Load cleaned dataset
# ============================================================

df = pd.read_parquet("persona_sjt_responses_clean_with_int.parquet").copy()

# normalize response if needed
if df["response_int"].min() == 1:
    df["y"] = df["response_int"] - 1
else:
    df["y"] = df["response_int"]

# ============================================================
# Means
# ============================================================

grand_mean = df["y"].mean()

persona_mean = df.groupby("persona_hash")["y"].mean()
scenario_mean = df.groupby("question_hash")["y"].mean()

# ============================================================
# Variance components
# ============================================================

persona_var = np.var(persona_mean - grand_mean, ddof=1)
scenario_var = np.var(scenario_mean - grand_mean, ddof=1)

total_var = np.var(df["y"], ddof=1)

residual_var = total_var - persona_var - scenario_var

# ============================================================
# Percent variance explained
# ============================================================

persona_pct = 100 * persona_var / total_var
scenario_pct = 100 * scenario_var / total_var
residual_pct = 100 * residual_var / total_var

results = pd.DataFrame({
    "component": ["persona", "scenario", "residual"],
    "variance": [persona_var, scenario_var, residual_var],
    "percent": [persona_pct, scenario_pct, residual_pct]
})

print("\nVariance decomposition:")
print(results)

results.to_csv("persona_scenario_variance_decomposition.csv", index=False)

print("\nSaved: persona_scenario_variance_decomposition.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# Load data
# ============================================================

df = pd.read_parquet("persona_sjt_responses_clean_with_int.parquet").copy()

if df["response_int"].min() == 1:
    df["y"] = df["response_int"] - 1
else:
    df["y"] = df["response_int"]

# ============================================================
# Compute observed persona variance
# ============================================================

grand_mean = df["y"].mean()
total_var = np.var(df["y"], ddof=1)

persona_mean = df.groupby("persona_hash")["y"].mean()
observed_persona_var = np.var(persona_mean - grand_mean, ddof=1)
observed_persona_pct = 100 * observed_persona_var / total_var

print("Observed persona variance:", observed_persona_var)
print("Observed persona percent:", observed_persona_pct)

# ============================================================
# Permutation test
# ============================================================

N_PERM = 500
null_vars = []

y = df["y"].to_numpy()
persona = df["persona_hash"].to_numpy()

for _ in range(N_PERM):
    shuffled = np.random.permutation(persona)

    tmp = pd.DataFrame({
        "persona": shuffled,
        "y": y
    })

    m = tmp.groupby("persona")["y"].mean()
    v = np.var(m - grand_mean, ddof=1)
    null_vars.append(v)

null_vars = np.array(null_vars)
null_pcts = 100 * null_vars / total_var

# ============================================================
# Compute p-value (finite-sample corrected)
# ============================================================

exceed = np.sum(null_vars >= observed_persona_var)
p_value = (exceed + 1) / (N_PERM + 1)

print("Permutation p-value:", p_value)

# ============================================================
# Plot
# ============================================================

plt.figure(figsize=(6, 4))
plt.hist(null_pcts, bins=30)
plt.axvline(observed_persona_pct, linestyle="--")

plt.xlabel("Persona variance explained under null (%)")
plt.ylabel("Frequency")
plt.title("Permutation test for persona variance")

plt.tight_layout()
plt.savefig("persona_variance_permutation.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd

# ============================================================
# Cell 3: Extract persona latent traits from idata_hexaco
# ============================================================

theta_post = idata_hexaco.posterior["theta"]

# Average over chains and draws
theta_mean = theta_post.mean(dim=("chain", "draw"))

# Convert to a tidy dataframe
theta_df = theta_mean.to_dataframe(name="theta").reset_index()

# Pivot to wide format: one row per person, one column per factor
theta_wide = theta_df.pivot(
    index="person",
    columns="factor",
    values="theta"
).reset_index()

# Attach persona hashes
theta_wide["persona_hash"] = person_index[theta_wide["person"].to_numpy()]

# Rename columns to prettier names if desired
rename_map = {
    "A": "Agreeableness",
    "C": "Conscientiousness",
    "E": "Emotionality",
    "X": "Extraversion",
    "H": "HonestyHumility",
    "O": "Openness",
}
theta_wide = theta_wide.rename(columns=rename_map)

# Keep only useful columns
final_cols = [
    "persona_hash",
    "Agreeableness",
    "Conscientiousness",
    "Emotionality",
    "Extraversion",
    "HonestyHumility",
    "Openness",
]
theta_wide = theta_wide[final_cols]

# Save
theta_wide.to_csv("persona_hexaco_latent_traits.csv", index=False)

print("Saved: persona_hexaco_latent_traits.csv")
print("\nTrait summary:")
print(theta_wide.describe(include="all"))
print("\nHead:")
print(theta_wide.head())

In [ ]:
import pandas as pd
import statsmodels.api as sm

# ------------------------------------------------
# Load datasets
# ------------------------------------------------

traits = pd.read_csv("persona_hexaco_latent_traits.csv")
responses = pd.read_parquet("persona_sjt_responses_clean_with_int.parquet")

# Convert responses to 0..5 if needed
if responses["response_int"].min() == 1:
    responses["y"] = responses["response_int"] - 1
else:
    responses["y"] = responses["response_int"]

# ------------------------------------------------
# Merge traits onto responses
# ------------------------------------------------

df = responses.merge(traits, on="persona_hash")

print("Merged rows:", len(df))

# ------------------------------------------------
# Create indicator: did persona choose option k?
# ------------------------------------------------

results = []

trait_cols = [
    "Agreeableness",
    "Conscientiousness",
    "Emotionality",
    "Extraversion",
    "HonestyHumility",
    "Openness",
]

for k, trait in enumerate(trait_cols):

    # binary outcome: chose option k
    y = (df["y"] == k).astype(int)

    X = df[trait_cols]
    X = sm.add_constant(X)

    model = sm.Logit(y, X).fit(disp=False)

    coef = model.params[trait]
    pval = model.pvalues[trait]

    results.append({
        "option": k,
        "trait": trait,
        "coef": coef,
        "p_value": pval
    })

results_df = pd.DataFrame(results)

print("\nTrait → Decision prediction:")
print(results_df)

results_df.to_csv("trait_decision_prediction.csv", index=False)

print("\nSaved: trait_decision_prediction.csv")

In [ ]:
! pip install statsmodels

In [ ]:
import pandas as pd

df = pd.read_parquet("persona_sjt_responses_clean_with_int.parquet")

if df["response_int"].min() == 1:
    df["y"] = df["response_int"] - 1
else:
    df["y"] = df["response_int"]

persona_var = df.groupby("persona_hash")["y"].mean().var()
scenario_var = df.groupby("question_hash")["y"].mean().var()

total_var = df["y"].var()

residual_var = total_var - persona_var - scenario_var

variance_df = pd.DataFrame({
    "component": ["persona", "scenario", "residual"],
    "variance": [persona_var, scenario_var, residual_var]
})

variance_df["percent"] = 100 * variance_df["variance"] / variance_df["variance"].sum()

print(variance_df)

variance_df.to_csv("variance_decomposition.csv", index=False)

print("\nSaved: variance_decomposition.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

theta_df = pd.read_csv("persona_hexaco_latent_traits.csv")

trait_names = [
    "Agreeableness",
    "Conscientiousness",
    "Emotionality",
    "Extraversion",
    "HonestyHumility",
    "Openness",
]

pretty_names = {
    "Agreeableness": "Agreeableness",
    "Conscientiousness": "Conscientiousness",
    "Emotionality": "Emotionality",
    "Extraversion": "Extraversion",
    "HonestyHumility": "Honesty-Humility",
    "Openness": "Openness",
}

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.flatten()

for i, col in enumerate(trait_names):
    axes[i].hist(theta_df[col], bins=25, edgecolor="black")
    axes[i].set_title(pretty_names[col])
    axes[i].set_xlabel("Latent trait value")
    axes[i].set_ylabel("Count")
    axes[i].axvline(theta_df[col].mean(), linestyle="--")

plt.tight_layout()
plt.savefig("hexaco_trait_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd

# Load your discrimination estimates
df = pd.read_csv("hexaco_discrimination_full_table.csv")

# Column assumed: a_mean (change if needed)
a = df["mean"]

# Define buckets
bins = [0, 0.5, 1.0, 2.0, 3.0, float("inf")]
labels = [
    "$a < 0.5$ (very weak)",
    "$0.5 \\le a < 1.0$ (weak--moderate)",
    "$1.0 \\le a < 2.0$ (good)",
    "$2.0 \\le a < 3.0$ (strong)",
    "$a \\ge 3.0$ (very strong)"
]

df["bucket"] = pd.cut(a, bins=bins, labels=labels, right=False)

# Count + percent
summary = (
    df["bucket"]
    .value_counts()
    .sort_index()
    .reset_index()
)
summary.columns = ["range", "count"]
summary["percent"] = 100 * summary["count"] / summary["count"].sum()

# Format for LaTeX
summary["percent"] = summary["percent"].round(1)

print(summary)

# Generate LaTeX table
latex = "\\begin{table}[h]\n\\centering\n\\begin{tabular}{lcc}\n\\hline\n"
latex += "Discrimination range & Number of items & Percentage \\\\\n\\hline\n"

for _, row in summary.iterrows():
    latex += f"{row['range']} & {int(row['count'])} & {row['percent']}\\% \\\\\n"

latex += "\\hline\n"
latex += f"Total & {summary['count'].sum()} & 100\\% \\\\\n"
latex += "\\hline\n\\end{tabular}\n"
latex += "\\caption{Distribution of item discrimination parameters estimated by the MIRT model.}\n"
latex += "\\label{tab:item_discrimination}\n\\end{table}"

print("\n" + latex)